# RideWise · Notebook 01 — Data Audit & Cleaning

**Meet the five RideWise tables, check they join cleanly, and record the single most important fact about this dataset.**

## The business question

Before any modelling, a data scientist has to answer a blunt question:
**can this data actually support the project?** RideWise wants to predict
churn. That is only possible if rider behaviour in the data is related to
whether riders leave. This notebook checks exactly that, and the answer
shapes every later decision.

### What you will learn
- How to load and profile a multi-table dataset
- How to verify referential integrity before you trust a join
- How to spot when a target variable carries no signal — and why that matters
- How to clean trip records with business-sensible filters

### How to read this notebook
Every section follows the same rhythm used throughout the project:
**the business question first**, then the data, then the method, then a
**validation check** that proves the step did what we claimed. This notebook
is **self-contained** — it defines the functions it needs and runs top to
bottom with no hidden state and no external project modules.

---

## 1. Libraries & configuration

In [1]:
# --- Environment Setup ---
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Project-wide constants used throughout the notebooks
RANDOM_STATE = 42                            # Set a random seed for reproducibility
CHURN_WINDOW_DAYS = 30                       # inactivity window that defines churn
LOYALTY_ORDER = {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3}            # Define loyalty tier order for sorting and analysis

# Where the raw CSVs live (this notebook sits in notebooks/, data in ../data/)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")        # Define the data directory path based on the current working directory

NAVY, ACCENT = "#1F3A5F", "#C8843C"                                         # Define color palette for plots
print("Libraries imported · data directory:", DATA_DIR.resolve())

Libraries imported · data directory: C:\Users\woro_\OneDrive\DS_Projects\Ride-hailing_Analytics_and_Churn_Prediction\data


## 2. Data loading

In [2]:
# ---- Loading a multi-table dataset ----

def load_raw(data_dir: Path = DATA_DIR) -> dict[str, pd.DataFrame]:
    """Load every raw CSV into a dict of DataFrames.

    Returns keys: riders, drivers, trips, sessions, promotions.
    """
    d = Path(data_dir)
    return {
        "riders":     pd.read_csv(d / "riders.csv"),
        "drivers":    pd.read_csv(d / "drivers.csv"),
        "trips":      pd.read_csv(d / "trips.csv"),
        "sessions":   pd.read_csv(d / "sessions.csv"),
        "promotions": pd.read_csv(d / "promotions.csv"),
    }

raw = load_raw()
for name, df in raw.items():
    print(f"{name:12s} {df.shape[0]:>7,} rows  x  {df.shape[1]:>2} cols")

riders        10,000 rows  x   8 cols
drivers        5,000 rows  x   7 cols
trips        200,000 rows  x  16 cols
sessions      50,000 rows  x   8 cols
promotions        20 rows  x  11 cols


**Overview of tables.** Examining real rows and cell contents, not just
column names. Numbers and formats tell you more than a schema does.

In [3]:
raw["riders"].head(3)

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN


In [4]:
raw["trips"].head(3)

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze


## 3. Referential integrity

A churn model joins trips and sessions onto riders. The three are related because riders initiate sessions to book trips. If trips referenced
riders that do not exist, those joins would silently drop data. **Check
before you trust.**

In [5]:
# ---- checking the referential columns and values for multi-table dataset to enable merging -----

riders, trips, sessions = raw["riders"], raw["trips"], raw["sessions"]

trip_match = trips["user_id"].isin(riders["user_id"]).mean()            # Calculate the proportion of trips that map to a known rider
sess_match = sessions["rider_id"].isin(riders["user_id"]).mean()        # Calculate the proportion of sessions that map to a known rider
print(f"Trips that map to a known rider:    {trip_match:.2%}")
print(f"Sessions that map to a known rider: {sess_match:.2%}")

assert trip_match == 1.0 and sess_match == 1.0, "Orphan records present!"          # Check that all trips and sessions have a parent rider; raise an error if not
print("\nValidation check PASSED — every trip and session has a parent rider.")

Trips that map to a known rider:    100.00%
Sessions that map to a known rider: 100.00%

Validation check PASSED — every trip and session has a parent rider.


## 4. The Target signal check

The `riders` table includes the `churn_prob` column. The obvious move is to
threshold it into a yes/no label and model that. But a label is only useful
if it **relates to the features**. Let's test whether `churn_prob` depends on
anything at all.

In [6]:
riders_1 = riders.copy()
riders_1["churn_flag"] = (riders_1["churn_prob"] > 0.5).astype(int)                     # Create churn flag based on churn probability

print("Correlation of churn_prob with numeric attributes:")
for col in ["age", "avg_rating_given"]:
    print(f"  {col:18s}: {riders_1[['churn_prob', col]].corr().iloc[0, 1]:+.3f}")       # Print correlation coefficient for each numeric attribute with churn_prob

print("\nMean churn flag by loyalty tier:")
print(riders_1.groupby("loyalty_status")["churn_flag"].mean().round(3))                 # Print mean churn flag by loyalty tier

print("\nMean churn flag by city:")
print(riders_1.groupby("city")["churn_flag"].mean().round(3))                           # Print mean churn flag by city

Correlation of churn_prob with numeric attributes:
  age               : -0.003
  avg_rating_given  : -0.001

Mean churn flag by loyalty tier:
loyalty_status
Bronze      0.108
Gold        0.097
Platinum    0.124
Silver      0.101
Name: churn_flag, dtype: float64

Mean churn flag by city:
city
Cairo      0.103
Lagos      0.106
Nairobi    0.110
Name: churn_flag, dtype: float64


### What this tells us

Every correlation is essentially **zero**, and churn does not vary by loyalty
tier or city. In plain terms: the supplied churn probability is **statistical
noise** — it is unrelated to who the rider is or how they behave. A model
trained to predict it would score no better than a coin flip (ROC-AUC ≈ 0.50).

> **This is the defining finding of the project.** It is handled transparently
> in feature engineering by constructing a *behavioural* churn label with realistic
> signal, and we document that step openly rather than pretending the raw
> data was learnable. Honesty about data quality is part of the craft.

## 5. Cleaning the records

Real logs contain impossible rows: zero fares, negative tips, trips that last
a fraction of a second or longer than a day. The two functions below apply
business-sensible filters and derive the fields later notebooks rely on.

In [7]:
def clean_trips(trips: pd.DataFrame) -> pd.DataFrame:
    """Parse timestamps, derive trip duration, drop impossible records."""
    t = trips.copy()
    t["pickup_time"]  = pd.to_datetime(t["pickup_time"],  utc=True, errors="coerce")            # Convert pickup_time to datetime, coerce errors to NaT
    t["dropoff_time"] = pd.to_datetime(t["dropoff_time"], utc=True, errors="coerce")            # Convert dropoff_time to datetime, coerce errors to NaT
    t["duration_min"] = (t["dropoff_time"] - t["pickup_time"]).dt.total_seconds() / 60.0        # Calculate trip duration in minutes#

    before = len(t)
    t = t[(t["fare"] > 0) & (t["tip"] >= 0)]                                # Keep only trips with positive fare and non-negative tip
    t = t[(t["duration_min"] > 0) & (t["duration_min"] < 24 * 60)]          # Keep only trips with positive duration and less than 24 hours
    t["surge_multiplier"] = t["surge_multiplier"].clip(lower=1.0)           # Clip surge_multiplier to a minimum of 1.0 (no surge)

    removed = before - len(t)
    if removed:
        print(f"clean_trips: removed {removed:,} invalid trip rows ({removed/before:.2%})")     # Print the number and percentage of removed invalid trip rows
    return t.reset_index(drop=True)


def clean_riders(riders: pd.DataFrame) -> pd.DataFrame:
    """Type the rider table and add a was_referred flag."""
    r = riders.copy()
    r["signup_date"]    = pd.to_datetime(r["signup_date"], utc=True, errors="coerce")         # Convert signup_date to datetime, coerce errors to NaT
    r["age"]            = r["age"].round().clip(lower=16, upper=99)                         # Clip ages to a reasonable range
    r["was_referred"]   = r["referred_by"].notna().astype(int)                              # Add a binary flag for whether the rider was referred (no data means no referral)
    r["loyalty_status"] = r["loyalty_status"].fillna("Bronze")                              # Fill missing loyalty_status with "Bronze"
    return r

In [8]:
trips_clean  = clean_trips(trips)
riders_clean = clean_riders(riders)

print("Trips after cleaning: ", f"{len(trips_clean):,}")
print("New trip columns:     ", [c for c in trips_clean.columns if c not in trips.columns])
print("New rider columns:    ", [c for c in riders_clean.columns if c not in riders.columns])
trips_clean[["fare", "surge_multiplier", "tip", "duration_min"]].describe().round(2)

output_path = '../data/trips_clean.csv' # Define output path for cleaned trips
trips_clean.to_csv(output_path, index=False) # Save cleaned trips to CSV
print(f"Saved cleaned trips to {output_path}")

output_path = '../data/riders_clean.csv' # Define output path for cleaned riders
riders_clean.to_csv(output_path, index=False) # Save cleaned riders to CSV
print(f"Saved cleaned riders to {output_path}")

Trips after cleaning:  200,000
New trip columns:      ['duration_min']
New rider columns:     ['was_referred']
Saved cleaned trips to ../data/trips_clean.csv
Saved cleaned riders to ../data/riders_clean.csv


In [9]:
trips_clean.head(3)

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,duration_min
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0


In [10]:
trips.dtypes

trip_id                 str
user_id                 str
driver_id               str
fare                float64
surge_multiplier    float64
tip                 float64
payment_type            str
pickup_time             str
dropoff_time            str
pickup_lat          float64
pickup_lng          float64
dropoff_lat         float64
dropoff_lng         float64
weather                 str
city                    str
loyalty_status          str
dtype: object

In [11]:
riders_clean.head(3)

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by,was_referred
0,R00000,2025-01-24 00:00:00+00:00,Bronze,35.0,Nairobi,5.0,0.142431,R00001,1
1,R00001,2024-09-09 00:00:00+00:00,Bronze,35.0,Nairobi,4.7,0.674161,NaN,0
2,R00002,2024-09-07 00:00:00+00:00,Bronze,47.0,Lagos,4.2,0.510379,NaN,0


In [12]:
riders.dtypes

user_id                 str
signup_date             str
loyalty_status          str
age                 float64
city                    str
avg_rating_given    float64
churn_prob          float64
referred_by             str
dtype: object

## 6. Snapshot & churn window (preview)

Notebook 03 builds features and the churn label in full. Here we just preview
the **snapshot design** that makes the later evaluation honest: freeze time,
build features from everything *before* the snapshot, and define churn by
whether the rider is active in the 30-day window *after* it.

In [13]:
def get_snapshot_date(trips: pd.DataFrame, window_days: int = CHURN_WINDOW_DAYS):
    """Split point between feature history and the forward label window."""
    return trips["pickup_time"].max() - pd.Timedelta(days=window_days) # function returns the snapshot date, which is the maximum pickup time minus the churn window in days

snapshot = get_snapshot_date(trips_clean)
hist   = trips_clean[trips_clean["pickup_time"] <= snapshot] # Feature history defined as trips that occurred on or before the snapshot date
future = trips_clean[trips_clean["pickup_time"] >  snapshot] # Future window defined as trips that occurred after the snapshot date

print("Snapshot date:        ", snapshot.date())
print(f"History-window trips:  {len(hist):,}")
print(f"Future-window trips:   {len(future):,}")
print("\nRiders active in the future window (i.e. NOT churned):",
      f"{future['user_id'].nunique():,} of {riders_clean['user_id'].nunique():,}") # Print the number of unique riders active in the future window compared to the total number of unique riders

Snapshot date:         2025-03-28
History-window trips:  183,682
Future-window trips:   16,318

Riders active in the future window (i.e. NOT churned): 8,007 of 10,000


## 7. Summary

- Five tables, all joining cleanly on rider id (referential integrity = 100%).
- 200k trips and 50k sessions give plenty of behavioural depth.
- **The raw churn target carries no signal** — the key constraint we design around.
- Trips and riders are cleaned with sanity filters; `duration_min` and
  `was_referred` are derived for later use.
- The snapshot / 30-day window design is previewed and ready for feature
  engineering.